In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StructField,
    StructType,
)

In [2]:
SENSOR_SCHEMA = StructType(
    [
        StructField("farm_sensor_id", IntegerType(), True),
        StructField("farm_id", IntegerType(), True),
        StructField("sensor_type_id", IntegerType(), True),
        StructField("value", DoubleType(), True),
        StructField("timestamp", LongType(), True),
    ]
)

In [4]:
def get_config():
    return {
        "kafka_bootstrap_servers": (
            f"{os.getenv('KAFKA_HOST', 'urbangreen-kafka')}:"
            f"{os.getenv('KAFKA_PORT_INTERNAL', '9092')}"
        ),
        "kafka_topic": os.getenv("KAFKA_TOPIC_SENSOR_READINGS", "sensor_readings"),
        "starting_offsets": os.getenv("KAFKA_STARTING_OFFSETS", "earliest"),
        "minio_endpoint": (
            f"http://{os.getenv('MINIO_HOST', 'urbangreen-minio')}:"
            f"{os.getenv('MINIO_API_PORT', '9000')}"
        ),
        "minio_access_key": os.getenv("MINIO_ROOT_USER", "minioadmin"),
        "minio_secret_key": os.getenv("MINIO_ROOT_PASSWORD"),
        "staging_bucket": os.getenv("MINIO_STAGING_BUCKET", "staging"),
        "trigger_interval": os.getenv("STREAM_TRIGGER_INTERVAL", "60 seconds"),
    }

In [6]:
config = get_config()
config

{'kafka_bootstrap_servers': 'urbangreen-kafka:9092',
 'kafka_topic': 'sensor_readings',
 'starting_offsets': 'earliest',
 'minio_endpoint': 'http://urbangreen-minio:9000',
 'minio_access_key': 'minioadmin',
 'minio_secret_key': 'minioadmin123',
 'staging_bucket': 'staging',
 'trigger_interval': '60 seconds'}

In [ ]:
def create_spark_session(config):
    return (
        # TODO
        SparkSession.builder.appName("sensor_readings_stream")
        .config("spark.hadoop.fs.s3a.endpoint", config["minio_endpoint"])
        .config("spark.hadoop.fs.s3a.access.key", config["minio_access_key"])
        .config("spark.hadoop.fs.s3a.secret.key", config["minio_secret_key"])
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
        )
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.streaming.schemaInference", "false")
        .getOrCreate()
    )

In [14]:
spark = create_spark_session(config)
spark.sparkContext.setLogLevel("WARN")

PySparkRuntimeError: [CONNECT_URL_NOT_SET] Cannot create a Spark Connect session because the Spark Connect remote URL has not been set. Please define the remote URL by setting either the 'spark.remote' option or the 'SPARK_REMOTE' environment variable.

In [15]:
def read_kafka_stream(spark, config):
    return (
        spark.readStream.format("kafka")
        .option(
            "kafka.bootstrap.servers",
            config["kafka_bootstrap_servers"],
        )
        .option(
            "subscribe",
            config["kafka_topic"],
        )
        .option(
            "startingOffsets",
            config["starting_offsets"],
        )
        .option(
            "failOnDataLoss",
            "false",
        )
        .load()
    )

In [16]:
kafka_df = read_kafka_stream(
    spark,
    config,
)

NameError: name 'spark' is not defined

In [18]:
def parse_sensor_readings(df):
    return (
        df.select(
            from_json(
                col("value").cast("string"),
                SENSOR_SCHEMA,
            ).alias("data")
        )
        .select("data.*")
        .withColumn(
            "event_date",
            to_date(from_unixtime(col("timestamp"))),
        )
    )

In [19]:
parsed_df = parse_sensor_readings(
    kafka_df,
)

NameError: name 'kafka_df' is not defined

In [20]:
def write_stream(df, config):
    output_path = f"s3a://{config['staging_bucket']}/raw/kafka/{config['kafka_topic']}/"

    checkpoint_path = (
        f"s3a://{config['staging_bucket']}/_checkpoints/kafka/{config['kafka_topic']}/"
    )

    return (
        df.writeStream.format("parquet")
        .outputMode("append")
        .option(
            "path",
            output_path,
        )
        .option(
            "checkpointLocation",
            checkpoint_path,
        )
        .partitionBy(
            "event_date",
        )
        .trigger(
            processingTime=config["trigger_interval"],
        )
        .start()
    )

In [21]:
query = write_stream(
    parsed_df,
    config,
)

query.awaitTermination()

NameError: name 'parsed_df' is not defined